# A Simple RAG Project

Before running the notebook, follow the setup instructions in [README.md](README.md).

In [ ]:

# Run once when setting up the project
#!git clone https://github.com/hodongtu/simple-rag-project
#%cd simple-rag-project
#!git pull

# Run once if packages are missing
# !pip install -r requirements.txt

### 1. Import Package

In [ ]:
import os
import sys
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv


W1029 17:44:30.704000 55866 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


### 2. Load environment variables & Set up the clients

In [ ]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add it to .env locally"
    )


client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("OPENAI_API_BASE"),
)

chat_model = os.getenv("MODEL_ECONOMY") or "gpt-4o-mini"

embedding_model = SentenceTransformer("BAAI/bge-m3")

print("Chat model:", chat_model)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [ ]:
# 4. Load knowledge base

with open("./data/knowledge.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Each paragraph becomes one chunk
chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print("Chunks:")
for c in chunks:
    print("-", c)


Chunks:
- Humans only use 2% of their brains, the Great Wall of China is visible from the Moon, drinking eight liters of water a day is essential for survival, and lightning never strikes the same place twice.
- Mount Everest is the tallest mountain on Earth. Its peak is 8848 meters above sea level and it is part of the Himalayas.
- The Amazon rainforest is the largest tropical rainforest in the world. It spans multiple countries in South America and contains immense biodiversity.
- The Pacific Ocean is the largest ocean on Earth. It covers more than 30% of the planet’s surface.


In [ ]:
# 5. Create embeddings

def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array(
    [get_embedding(chunk) for chunk in chunks]
).astype("float32")
embedding_dim = chunk_embeddings.shape[1]



# 6. Create FAISS index

index = faiss.IndexFlatIP(embedding_dim)
# remove previously added content
index.reset()
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)


FAISS index size: 4


In [ ]:
# 7. User question

question = "Con người sử dụng bao nhiêu % năng lực của bộ não?"
print("Question:", question)


# 8. Embed the question

question_embedding = np.array(
    [get_embedding(question)]
).astype("float32")

Question: Con người sử dụng bao nhiêu % năng lực của bộ não?


In [ ]:
# ==============================
# 9. Retrieve top 2 chunks
# ==============================

k = 2

distances, indices = index.search(question_embedding, k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved context:")
for c in retrieved_chunks:
    print("-", c)


Retrieved context:
- Humans only use 2% of their brains, the Great Wall of China is visible from the Moon, drinking eight liters of water a day is essential for survival, and lightning never strikes the same place twice.
- The Pacific Ocean is the largest ocean on Earth. It covers more than 30% of the planet’s surface.


In [ ]:
# ==============================
# 10. Create RAG prompt
# ==============================

context = "\n\n".join(retrieved_chunks)

prompt = f"""
# Instructions
Answer the following question in one short response.
Use only the context provided, even if it contains false facts.
Say "There is no information awailable" if there is no answer in the context

# Context:
{context}

# Question:
{question}

# Answer:
"""

In [ ]:
# ==============================
# 11. Call GPT-4o to generate the answer
# ==============================

response = client.chat.completions.create(
    model=chat_model,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print("\nLLM Answer:")
print(response.choices[0].message.content)


LLM Answer:
Con người chỉ sử dụng 2% năng lực của bộ não.
